# Bước 1: Thống nhất & Kiểm tra Dữ liệu Bàn giao từ Ku em 1

**Mục tiêu của Notebook này:**
1. Nạp dữ liệu đã tiền xử lý từ Ku em 1 tại `data/processed/results/processed_data/` (`processed_graph.pt`, `train_pairs.pt`, `val_pairs.pt`, `test_pairs.pt`).
2. Tổng hợp đồ thị đồng nhất trên `50,000` nút `customer` (`x`, `edge_index`, `node_id_map`) để phục vụ hai mô hình đối chứng **GCN thuần** và **GAT thuần**.
3. Làm rõ bản chất các kênh cạnh trong `edge_index` và kiểm tra **chống rò rỉ dữ liệu (Edge Leakage Check)**.
4. In đầy đủ thống kê: số nút, số cạnh, kích thước đặc trưng và số mẫu dương/âm (kèm 5 loại mẫu âm `Type 1..5`) của từng tập `Train / Val / Test`.

## 1.1. Khai báo thư viện & Đường dẫn dữ liệu

In [1]:
import os
import sys
import torch
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_validator import (
    DEFAULT_PROCESSED_DIR,
    NEGATIVE_TYPE_NAMES,
    build_homogeneous_customer_graph,
    load_split_pairs,
    check_edge_leakage,
    inspect_graph_data,
)

print("Thư mục dữ liệu của Ku em 1:", DEFAULT_PROCESSED_DIR)

Thư mục dữ liệu của Ku em 1: C:\KOUPC\study\5th\DSS\CK\data\processed\results\processed_data


## 1.2. Nạp đồ thị gốc của Ku em 1 & Chuyển sang Đồ thị Đồng nhất Khách hàng (`x`, `edge_index`)

In [2]:
graph_dict = build_homogeneous_customer_graph(DEFAULT_PROCESSED_DIR, include_covert_channels=True)
splits = load_split_pairs(DEFAULT_PROCESSED_DIR)

print("=== Cấu trúc các loại đỉnh gốc trong processed_graph.pt ===")
for node_type, shape in graph_dict["raw_hetero_summary"].items():
    print(f"  - {node_type:12s}: shape = {shape}")

print("\n=== Ánh xạ ID gốc <-> Chỉ số nút (5 mẫu đầu tiên) ===")
for idx in range(5):
    print(f"  Node index {idx} <-> {graph_dict['node_id_map'][idx]}")

=== Cấu trúc các loại đỉnh gốc trong processed_graph.pt ===
  - customer    : shape = (50000, 10)
  - account     : shape = (112620, 6)
  - device      : shape = (167286, 6)
  - ip          : shape = (166933, 4)
  - beneficiary : shape = (644166, 8)
  - bank        : shape = (4, 4)

=== Ánh xạ ID gốc <-> Chỉ số nút (5 mẫu đầu tiên) ===
  Node index 0 <-> CUST_00000
  Node index 1 <-> CUST_00001
  Node index 2 <-> CUST_00002
  Node index 3 <-> CUST_00003
  Node index 4 <-> CUST_00004


## 1.3. Thống kê số nút, số cạnh, kích thước đặc trưng và số mẫu Dương/Âm từng tập

In [3]:
graph_summary_df, split_summary_df = inspect_graph_data(graph_dict, splits)

print("=== 1. BẢNG THỐNG KÊ ĐỒ THỊ ĐẦU VÀO (GCN / GAT) ===")
display(graph_summary_df)

print("\n=== 2. BẢNG PHÂN BỔ MẪU DƯƠNG (y=1) & 5 LOẠI MẪU ÂM (y=0) TRÊN TRAIN / VAL / TEST ===")
display(split_summary_df)

print("\n=== 3. Ý NGHĨA CÁC NHÃN VÀ 5 PHÂN LỚP MẪU ÂM (HARD NEGATIVES) ===")
for k, desc in NEGATIVE_TYPE_NAMES.items():
    print(f"  edge_type = {k}: {desc}")

=== 1. BẢNG THỐNG KÊ ĐỒ THỊ ĐẦU VÀO (GCN / GAT) ===

=== 2. BẢNG PHÂN BỔ MẪU DƯƠNG (y=1) & 5 LOẠI MẪU ÂM (y=0) TRÊN TRAIN / VAL / TEST ===

=== 3. Ý NGHĨA CÁC NHÃN VÀ 5 PHÂN LỚP MẪU ÂM (HARD NEGATIVES) ===
  edge_type = 0: Positive (Đồng phạm cùng kịch bản - y=1)
  edge_type = 1: Type 1: AML-AML (Khác kịch bản - y=0)
  edge_type = 2: Type 2: AML-Clean (Tội phạm - Người sạch - y=0)
  edge_type = 3: Type 3: Suspected KYC (Nghi vấn giả - y=0)
  edge_type = 4: Type 4: Shared IP/Device (Người sạch chung mạng - y=0)
  edge_type = 5: Type 5: Clean-Clean (Ngẫu nhiên hoàn toàn - y=0)


Chi_so,Gia_tri
Số lượng nút khách hàng (num_nodes),50000
Kích thước đặc trưng nút (num_features),60
Tổng số cạnh truyền tin sau hợp nhất (edge_index),557062
├─ Kênh RELATED_TO (KYC 2 chiều),62262
├─ Kênh SHARED_DEVICE (Dùng chung thiết bị),2676
├─ Kênh SHARED_IP (Dùng chung IP),3218
└─ Kênh TRANSACTS (Giao dịch lớn/nhiều lần),490110
Số giá trị NaN trong x,0
Số giá trị Inf trong x,0


Split,Total_Pairs,Positive (y=1),Negative (y=0),Type1_AML_AML,Type2_AML_Clean,Type3_Suspected,Type4_Shared_IP_Dev,Type5_Clean_Clean
TRAIN,12664,6332,6332,1899,949,949,496,2039
VAL,2302,1151,1151,345,172,172,17,445
TEST,3420,1710,1710,513,256,256,32,653


## 1.4. Kiểm tra chống rò rỉ dữ liệu (Edge Leakage & Scenario Split Verification)

- **Làm rõ bản chất cạnh:** Quan hệ mục tiêu cần dự đoán (`y=1`) là **quan hệ đồng phạm rửa tiền (`AML Accomplice` từ `labels.json`)**, trong khi `edge_index` đầu vào chỉ chứa các quan hệ khai báo KYC (`RELATED_TO`) và kênh hành vi quan sát được (`SHARED_DEVICE`, `SHARED_IP`, `TRANSACTS`).
- Kiểm tra xác nhận **0%** cạnh đồng phạm mục tiêu xuất hiện trong `RELATED_TO` và **0%** giao nhau về tập đỉnh đồng phạm giữa `Train`, `Val`, `Test` (Scenario Split).

In [4]:
leak_check = check_edge_leakage(graph_dict, splits)
leak_df = pd.DataFrame([{"Kiem_tra": k, "Ket_qua": v} for k, v in leak_check.items()])
display(leak_df)
assert leak_check["leakage_free"], "Phát hiện rò rỉ cạnh đích trong RELATED_TO!"
print("\n[HOÀN THÀNH BƯỚC 1] Dữ liệu hợp lệ, không NaN/Inf và đảm bảo chống rò rỉ dữ liệu!")


[HOÀN THÀNH BƯỚC 1] Dữ liệu hợp lệ, không NaN/Inf và đảm bảo chống rò rỉ dữ liệu!


Kiem_tra,Ket_qua
train_pos_in_RELATED_TO,0
val_pos_in_RELATED_TO,0
test_pos_in_RELATED_TO,0
train_val_pos_node_overlap,0
train_test_pos_node_overlap,0
val_test_pos_node_overlap,0
leakage_free,True
